# 🚀 Instalación y Configuración de OpenSpec (PowerShell)

Notebook estructurado para instalar y configurar el stack completo de herramientas de desarrollo usando **PowerShell**.

**Estructura:**
1. **Verificación de dependencias** (Python, Node, npm, Git, uv)
2. **Instalación de herramientas globales** (Serena, Graphify, Codeburn, OpenSpec, OpenSpec UI, Claude History Viewer, CodeGraph)
3. **Inicialización del proyecto** (submódulos, Serena, CodeGraph, OpenSpec, copia de `lidr-specboot`)

---
⚠️ **Requisitos previos:**
- Windows con PowerShell 7+
- Jupyter Notebook instalado
- Conexión a internet
- Si usas windows y VSCode instala el plugin: [Polyglot Notebooks extension](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode)

## ⚙️ Configuración inicial

In [ ]:
# Variables globales
$ErrorActionPreference = "Stop"
$Force = $false  # Cambiar a $true para forzar reinstalaciones

# Crear directorio de logs
$LogDir = Join-Path $PWD "logs"
New-Item -ItemType Directory -Force -Path $LogDir | Out-Null

Write-Host "✅ Entorno configurado. Force = $Force" -ForegroundColor Green

---
## 🧩 Etapa 1 — Verificación de dependencias

In [ ]:
# Versiones mínimas requeridas
$MinVersions = @{
    python = "3.13.0"
    node   = "20.0.0"
    npm    = "10.0.0"
    git    = "2.40.0"
    uv     = "0.7.0"
}

function Get-VersionValue([string]$tool) {
    switch ($tool) {
        "python" { return (python --version).Split()[1] }
        "node"   { return ((node --version).TrimStart("v")) }
        "npm"    { return (npm --version) }
        "git"    { return ((git --version).Split()[2]) }
        "uv"     { return ((uv --version).Split()[1]) }
    }
}

function Assert-Command([string]$cmd, [string]$min) {
    if (-not (Get-Command $cmd -ErrorAction SilentlyContinue)) {
        throw "❌ $cmd not found in PATH."
    }
    $v = Get-VersionValue $cmd
    # Extraer solo major.minor.patch (ej: 2.47.1.windows.1 → 2.47.1)
    $vClean = ($v -replace '^(\d+\.\d+\.\d+).*', '$1')
    if ([version]$vClean -lt [version]$min) {
        throw "❌ $cmd version $v is below required $min"
    }
    Write-Host "✅ $cmd $v OK" -ForegroundColor Green
}

# Verificar todas las dependencias
foreach ($k in $MinVersions.Keys) {
    Assert-Command $k $MinVersions[$k]
}

---
## 🛠️ Etapa 2 — Instalación de herramientas globales

### 2.1 Instalar Serena

In [ ]:
if ((Get-Command serena -ErrorAction SilentlyContinue) -and -not $Force) {
    Write-Host "ℹ️  serena already installed." -ForegroundColor Yellow
} else {
    Write-Host "📦 Installing serena..." -ForegroundColor Cyan
    uv tool install -p 3.13 serena-agent
}

### 2.2 Instalar Graphify

In [ ]:
if ((Get-Command graphify -ErrorAction SilentlyContinue) -and -not $Force) {
    Write-Host "ℹ️  graphify already installed." -ForegroundColor Yellow
} else {
    Write-Host "📦 Installing graphify..." -ForegroundColor Cyan
    uv tool install graphify
    graphify install --platform windows
}

### 2.3 Instalar Codeburn

In [ ]:
if ((Get-Command codeburn -ErrorAction SilentlyContinue) -and -not $Force) {
    Write-Host "ℹ️  codeburn already installed." -ForegroundColor Yellow
} else {
    Write-Host "📦 Installing codeburn..." -ForegroundColor Cyan
    npm install -g codeburn
}

### 2.4 Instalar OpenSpec

In [ ]:
if ((Get-Command openspec -ErrorAction SilentlyContinue) -and -not $Force) {
    Write-Host "ℹ️  openspec already installed." -ForegroundColor Yellow
} else {
    Write-Host "📦 Installing OpenSpec..." -ForegroundColor Cyan
    npm install -g @fission-ai/openspec@latest
}

### 2.5 Instalar OpenSpec UI

In [ ]:
if ((Get-Command openspecui -ErrorAction SilentlyContinue) -and -not $Force) {
    Write-Host "ℹ️  openspecui already installed." -ForegroundColor Yellow
} else {
    Write-Host "📦 Installing OpenSpec UI..." -ForegroundColor Cyan
    npm install -g openspecui
}

### 2.6 Instalar Claude History Viewer

In [ ]:
# Ruta donde se instala el programa (menú de inicio del usuario)
$startMenuPath = Join-Path $env:LOCALAPPDATA "Claude Code History Viewer"
$shortcutPath = Join-Path $startMenuPath "claude-code-history-viewer.exe"

# Validar si ya está instalado buscando la carpeta o el acceso directo
$isInstalled = (Test-Path $startMenuPath) -or (Test-Path $shortcutPath)


Write-Host $shortcutPath

if ($isInstalled -and -not $Force) {
    Write-Host "ℹ️  Claude Code History Viewer already installed at: $startMenuPath" -ForegroundColor Yellow
} else {
    if ($isInstalled -and $Force) {
        Write-Host "🗑️  Removing existing installation..." -ForegroundColor Yellow
        # Desinstalar usando el desinstalador si existe
        $uninstaller = Join-Path $startMenuPath "Uninstall Claude Code History Viewer.lnk"
        if (Test-Path $uninstaller) {
            Start-Process -FilePath $uninstaller -ArgumentList "/S" -Wait
        }
        Remove-Item -Recurse -Force $startMenuPath -ErrorAction SilentlyContinue
    }
    
    Write-Host "📦 From https://github.com/jhlee0409/claude-code-history-viewer" -ForegroundColor Cyan
    
    # Obtener la última release desde la API de GitHub
    $release = irm "https://api.github.com/repos/jhlee0409/claude-code-history-viewer/releases/latest"

    # Filtrar el asset que quieres (para Windows x64)
    $asset = $release.assets | Where-Object { $_.name -like "*x64-setup.exe" } | Select-Object -First 1

    if (-not $asset) {
        throw "No se encontró el instalador para Windows x64"
    }

    # Descargar el archivo
    $downloadPath = Join-Path $env:TEMP $asset.name
    Write-Host "📥 Descargando $($asset.name)..." -ForegroundColor Cyan
    irm $asset.browser_download_url -OutFile $downloadPath

    # Ejecutar el instalador silenciosamente
    Write-Host "🔧 Instalando silenciosamente..." -ForegroundColor Cyan
    Start-Process -FilePath $downloadPath -ArgumentList "/S" -Wait

    # Verificar que se instaló correctamente
    Start-Sleep -Seconds 210
    if (Test-Path $startMenuPath) {
        Write-Host "✅ Instalación completada en: $startMenuPath" -ForegroundColor Green
    } else {
        Write-Warning "⚠️  La instalación puede haber fallado. Verifica manualmente."
    }

    # Limpiar
    Remove-Item $downloadPath -Force
}

### 2.7 Instalar CodeGraph

In [ ]:
if (-not (Get-Command codegraph -ErrorAction SilentlyContinue) -or $Force) {
    Write-Host "📦 Installing CodeGraph..." -ForegroundColor Cyan
    irm https://raw.githubusercontent.com/colbymchenry/codegraph/main/install.ps1 | iex
} else {
    Write-Host "ℹ️  codegraph already installed." -ForegroundColor Yellow
}

---
## 🏗️ Etapa 3 — Inicialización del proyecto

In [ ]:
# === Parámetros del proyecto (EDITA AQUÍ) ===
$ProjectPath = "C:\dev\customers\arcteryx\platform-reservation-api\"  # 👈 Ruta absoluta al repositorio
$ProjectName = "platform-reservation-api"    # 👈 Nombre lógico del proyecto

if (-not (Test-Path $ProjectPath)) {
    throw "❌ ProjectPath does not exist: $ProjectPath"
}

Set-Location $ProjectPath

if (-not (Test-Path ".git")) {
    throw "❌ The target directory is not a Git repository."
}

Write-Host "📂 Working in: $(Get-Location)" -ForegroundColor Green

### 3.1 Agregar submódulo `dev-standards`

In [ ]:
if (-not (Test-Path ".gitmodules") -or $Force) {
    if (-not (Test-Path "external/dev-standards")) {
        Write-Host "📦 Adding submodule dev-standards..." -ForegroundColor Cyan
        git submodule add https://github.com/enterprise-repo/dev-standards.git external/dev-standards
    } else {
        Write-Host "ℹ️  Submodule directory already exists." -ForegroundColor Yellow
    }
}

Write-Host "🔄 Initializing submodules..." -ForegroundColor Cyan
git submodule update --init --recursive

### 3.2 Inicializar Serena

In [ ]:
Write-Host "📝 Creating Serena project: $ProjectName" -ForegroundColor Cyan
serena project create --name $ProjectName
$createResult = $LASTEXITCODE

if ($createResult -ne 0) {
    Write-Warning "⚠️  serena project create failed (exit code: $createResult)"
    Write-Host "🔄 Falling back to: serena project index" -ForegroundColor Yellow
    serena project index
    
    if ($LASTEXITCODE -ne 0) {
        throw "❌ Both serena project create and serena project index failed"
    }
    Write-Host "✅ serena project index completed successfully" -ForegroundColor Green
} else {
    Write-Host "✅ serena project create completed successfully" -ForegroundColor Green
}

### 3.3 Inicializar CodeGraph

In [ ]:
if (-not (Test-Path ".codegraph") -or $Force) {
    Write-Host "🔧 Initializing CodeGraph..." -ForegroundColor Cyan
    codegraph init -i
} else {
    Write-Host "ℹ️  CodeGraph already initialized." -ForegroundColor Yellow
}

### 3.4 Inicializar OpenSpec

In [ ]:
#corre esto directamente en la terminal 
Set-Location "Path real" #"C:\dev\customers\arcteryx\platform-reservation-api"

Write-Host "🔧 Initializing OpenSpec..." -ForegroundColor Cyan
openspec init --tools kiro #--tolls is optional

#openspec tiene dos workflows core y custom pero si seleccionamos custom necesitamos correrlo en la terminal 
#el siguiente comando para que se configure correctamente el workflow patterns
Write-Host "🔧 Configuring OpenSpec profile...for Workflow Patterns (Expanded Mode)" -ForegroundColor Cyan
openspec config profile
    
Write-Host "🔄 Updating OpenSpec..." -ForegroundColor Cyan
openspec update

### 3.5 Copiar skills adicionales de repositorio

En este caso por medio del tool npm skills instalaremos diractemente los skills de: 

- adversarial—review
- code—auditing
- commit
- enrich—us
- explain
- meta—prompt
- owasp—security—audit
- run—parallel—tasks
- show—spec—working
- sync—agent—symlinks
- update—docs
- using—git—worktrees
- writing—skills

desde el repositorio git:

robertoivanmezatrillo/ai-skills

In [ ]:
 npx skills add robertoivanmezatrillo/ai-skills --skill '*' -y -a kiro-cli -g

---
## ✅ Resumen final

In [ ]:
Set-Location $ProjectPath
Write-Host "`n🎉 Installation and configuration completed successfully!" -ForegroundColor Green
Write-Host "`n🔧 Global tools status:" -ForegroundColor Cyan

$tools = @("serena", "graphify", "codeburn", "openspec", "openspecui", "codegraph")
foreach ($tool in $tools) {
    if (Get-Command $tool -ErrorAction SilentlyContinue) {
        Write-Host "  ✔ $tool" -ForegroundColor Green
    } else {
        Write-Host "  ✘ $tool" -ForegroundColor Red
    }
}

Write-Host "`n📂 Project configured in: $(Get-Location)" -ForegroundColor Green